In [ ]:
import os
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

from napistu.gcs import downloads
from napistu.sbml_dfs_core import SBML_dfs
from napistu import utils as napistu_utils

# globals
DATA_DIR = "napistu_data"
ASSET = "human_consensus"
VERSION_TAG = "20250916"

# download the Octopus sbml_dfs and the other assets its bundled with
sbml_dfs_path = downloads.load_public_napistu_asset(
    asset = ASSET,
    subasset = "sbml_dfs",
    data_dir = DATA_DIR,
    version = VERSION_TAG,
    overwrite = False
)


In [7]:
# functions

def simple_pd_heatmap(df, plot_title):
    # Create the heatmap
    plt.figure(figsize=(10, 8))
    sns.clustermap(df, 
                annot=True,  # Show values in cells
                cmap='Blues', 
                fmt='d',  # Integer format for annotations
                cbar_kws={'label': "Counts"})
    plt.title(plot_title)
    plt.tight_layout()
    plt.show()

In [3]:
sbml_dfs = SBML_dfs.from_pickle(sbml_dfs_path)


In [ ]:
# net summary
sbml_dfs.show_summary()

In [4]:
species_ontology_counts_by_bqb = (
    sbml_dfs.get_ontology_occurrence(
        "species",
        stratify_by_bqb=True,
        allow_col_multindex = True,
        characteristic_only=True,
        dogmatic=False
    )
    .sum()
    .unstack(fill_value=0)
)

pivoted_species_ontology_counts_by_bqb = (
    species_ontology_counts_by_bqb
    .loc[species_ontology_counts_by_bqb.sum(axis=1) > 500]
    .loc[lambda x: x.sum(axis=1).sort_values(ascending=False).index]
)

napistu_utils.show(pivoted_species_ontology_counts_by_bqb)

bqb,BQB_HAS_PART,BQB_IS,BQB_IS_ENCODED_BY,BQB_IS_HOMOLOG_TO
ontology,,,,
reactome,0,25928,0,141573
ensembl_transcript,43,104,160804,0
uniprot,11259,121048,0,0
ensembl_protein,0,104583,0,0
intact,0,27334,0,0
ensembl_gene,807,809,22726,0
ncbi_entrez_gene,0,76,20310,0
chebi,1600,5801,0,0
refseq_synonym,0,5612,0,0


In [6]:
species_source_cooccurrence = (
    sbml_dfs.get_source_cooccurrence("species")
    .rename_axis('Database', axis=0)
    .rename_axis('Database', axis=1)
)

simple_pd_heatmap(species_source_cooccurrence, "Species Source Co-occurrence")

NameError: name 'simple_pd_heatmap' is not defined

In [ ]:
species_source_cooccurrence.rename_axis('Database', axis=0).rename_axis('Database', axis=1)

In [ ]:
species_source_cooccurrence

In [ ]:
species_ontology_by_source_cooccurrence = sbml_dfs.get_ontology_x_source_cooccurrence(
    "species",
    stratify_by_bqb=False,
    characteristic_only=True,
    dogmatic=False
)

filtered_species_ontology_by_source_cooccurrence = (
    species_ontology_by_source_cooccurrence
    .loc[species_ontology_by_source_cooccurrence.index.isin(species_ontology_counts.index[0:12].tolist())]
)

simple_pd_heatmap(filtered_species_ontology_by_source_cooccurrence, "Count", "Source x Ontology Co-occurrence")
